# Final run: train P1-P3 once more, then the official test

P1 = `p1f_ctl`, P2 = `p2c_plain`, P3 = prototypes on the final P2 weights. Seeds 42, 43, 44.
One run does it all: continue each trained model (`epoch_10.pt`) for 2 epochs on all 8,000 training images
(backbone LR 1e-5, head LR 1e-4), check that the training loss went down, fix the models, then score the official test once.

**Inputs:** the code dataset with `cdc-cp_code_final.zip`, the DPD-Small dataset, the P1f notebook output and the P2e notebook output.

**Run it twice (GPU T4 x2, Internet on):**
1. `MODE = 'smoke'`: Run All. A short check of every step on the validation sets. It never reads the official test CSVs.
2. `MODE = 'final'`: Save Version -> Save & Run All. **Only once.** The results are in `final/test_results/`.

## 1. Settings

In [ ]:
from pathlib import Path

MODE = 'smoke'   # 'smoke' first, then 'final' (once)
assert MODE in ('smoke', 'final'), MODE
SMOKE = MODE == 'smoke'
WORKERS = 2
SEEDS = (42,) if SMOKE else (42, 43, 44)
PARENTS = ['p1f_ctl', 'p2c_plain']   # P1 and P2; P3 reuses the P2 weights
PARENT_EPOCH = 10
INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/final_smoke' if SMOKE else '/kaggle/working/final')
LOCK_DIR = OUT / 'lock'
CODE_DIR = Path('/tmp/cdc-cp')
DATA_DIR = Path('/tmp/DPD_Small')
STANDIN_DIR = Path('/tmp/DPD_standin')   # smoke only: stand-in "official" CSVs made from the validation sets
CACHE_DIR = Path('/tmp/cache')

## 2. GPUs and code

In [ ]:
import hashlib, os, shutil, subprocess, sys, zipfile, time
import torch

assert torch.cuda.is_available(), 'Settings > Accelerator > GPU T4 x2'
N_GPU = torch.cuda.device_count()
print(N_GPU, 'x', torch.cuda.get_device_name(0), '| torch', torch.__version__)

os.chdir('/kaggle/working')  # a rerun deletes CODE_DIR, so the kernel must not be inside it
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
NEEDS = {'cdccp/data.py': 'def upright', 'cdccp/stage2.py': 'def run_stage2', 'cdccp/final.py': 'def verify_lock', 'scripts/final_test.py': '--confirm-final-test',
         'data/mappings/catalogue_final.json': '{'}


def is_final_code(root):
    return all((root / f).is_file() and text in (root / f).read_text(encoding='utf-8') for f, text in NEEDS.items())


def code_digest(root):
    files = sorted(p for d in ('cdccp', 'scripts', 'configs') for p in (root / d).rglob('*') if p.suffix in ('.py', '.yaml'))
    return hashlib.sha256(b''.join(p.relative_to(root).as_posix().encode() + p.read_bytes() for p in files)).hexdigest()


def pick_code(input_dir, unzip_to):
    """The one final-code folder among the inputs. Older code folders are skipped; two different final folders are refused."""
    if unzip_to.exists():
        shutil.rmtree(unzip_to)
    unzip_to.mkdir(parents=True)
    for i, z in enumerate(sorted(input_dir.rglob('cdc-cp_code*.zip'))):
        with zipfile.ZipFile(z) as archive:
            archive.extractall(unzip_to / str(i))
    folders = sorted({p.parent.parent for d in (input_dir, unzip_to) for p in d.rglob('cdccp/__init__.py')})
    good = [f for f in folders if is_final_code(f)]
    for f in folders:
        print('code folder:', f, '(final)' if f in good else '(old code, skipped)')
    assert good, f'No final code among the inputs {folders}: upload cdc-cp_code_final.zip as a new version of the code dataset'
    digests = {code_digest(f) for f in good}
    assert len(digests) == 1, f'{len(digests)} different final code folders {good}: keep only the newest cdc-cp_code_final.zip'
    return good[0]


found = pick_code(INPUT, Path('/tmp/code_zips'))
shutil.copytree(found, CODE_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)
os.chdir(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
print('code from', found)

## 3. Helpers (definitions only; the next cells use them)

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
import pandas as pd
import yaml
from cdccp.checkpoint import file_sha256, load_checkpoint
from cdccp.config import load_config
from cdccp.data import COLUMNS


def sh(*args, env=None):
    proc = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=env)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'exit code {proc.returncode}: {args}')


def is_smoke(path):
    """True for anything under a smoke output folder (p1f_smoke, p2e_smoke, final_smoke)."""
    return any(part.endswith('_smoke') for part in Path(path).parts)


def link_root(root, train_csv, image_src):
    """A data root with the training CSV and a link to the image folder."""
    root.mkdir(parents=True, exist_ok=True)
    shutil.copy2(train_csv, root / 'DPD_Small_train.csv')
    link = root / 'DPD_Small_dataset'
    if not link.exists():
        link.symlink_to(image_src, target_is_directory=True)
    return link


def write_standins(root, splits_dir):
    """Smoke only: the development validation sets in the official CSV format (no header), under the official names."""
    for name, split in (('test_seen', 'val_seen'), ('test_unseen', 'val_unseen')):
        df = pd.read_csv(Path(splits_dir) / f'{split}.csv')
        df[COLUMNS].to_csv(root / f'DPD_Small_{name}.csv', header=False, index=False)


def find_parent(config, seed, input_dir, epoch=10):
    """The parent epoch checkpoint and its dev board. Smoke outputs are skipped. Copies must be byte-identical."""
    run = f'{config}_seed{seed}'
    hits = [p for p in sorted(Path(input_dir).rglob(f'{run}/epoch_{epoch:02d}.pt')) if not is_smoke(p)]
    assert hits, f'no {run}/epoch_{epoch:02d}.pt in the inputs: add the P1f and P2e notebook outputs as inputs'
    shas = {file_sha256(p) for p in hits}
    assert len(shas) == 1, f'{run}: {len(hits)} different epoch_{epoch:02d}.pt copies {hits}'
    parent = hits[0]
    meta = load_checkpoint(parent)
    got = (meta['config']['name'], meta['seed'], meta['epoch'])
    assert got == (config, seed, epoch), f'{parent}: holds {got}, expected {(config, seed, epoch)}'
    board = parent.parent / f'p3_board_ep{epoch:02d}.csv'
    assert board.is_file(), f'{run}: no development board {board.name} beside the checkpoint'
    return parent, board


def stage2_job(config, seed, parent, lock_dir, smoke):
    name = f'{config}_seed{seed}'
    args = [sys.executable, 'scripts/stage2.py', parent, lock_dir / name, f'paths.cache_dir={CACHE_DIR}', f'loader.num_workers={WORKERS}']
    if smoke:
        args += ['stage2.epochs=1', 'stage2.train_split=train_fit']
    return name, args


def run_queue(jobs, log_dir):
    """Run (name, args) jobs, one per GPU at a time. Returns the names that failed."""
    log_dir.mkdir(parents=True, exist_ok=True)
    waiting, running, failed = list(jobs), {}, []
    print('queue:', [n for n, _ in waiting], flush=True)
    while waiting or running:
        for gpu in range(max(N_GPU, 1)):
            if gpu not in running and waiting:
                name, args = waiting.pop(0)
                log = open(log_dir / f'{name}.log', 'a')
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
                p = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, env=env, stdout=log, stderr=subprocess.STDOUT, text=True)
                running[gpu] = (name, p, log, time.time())
                print(f'[gpu {gpu}] start {name}', flush=True)
        time.sleep(20)
        for gpu, (name, p, log, t0) in list(running.items()):
            if p.poll() is not None:
                log.close()
                del running[gpu]
                lines = (log_dir / f'{name}.log').read_text(errors='replace').splitlines()
                if p.returncode:
                    failed.append(name)
                    print(f'[gpu {gpu}] FAILED {name}; last lines:', *lines[-15:], sep='\n', flush=True)
                else:
                    print(f'[gpu {gpu}] finished {name} in {(time.time() - t0) / 60:.0f} min:', *lines[-2:], sep='\n  ', flush=True)
    return failed


def gate_table(lock_dir, names):
    """One row per Stage 2 run: PASS/FAIL, the reason and the train loss of each epoch."""
    rows = []
    for name in names:
        gate = json.loads((lock_dir / name / 'stage2_gate.json').read_text())
        rows.append({'run': name, 'gate': 'PASS' if gate['ok'] else 'FAIL', 'reason': gate['reason'],
                     **{f'loss_ep{i + 1}': round(v, 4) for i, v in enumerate(gate['train_loss_by_epoch'])}})
    return pd.DataFrame(rows)


KINDS = {'p1f_ctl': [('P1', 'head')], 'p2c_plain': [('P2', 'head'), ('P3pooled', 'proto-pooled'), ('P3balanced', 'proto-balanced')]}


def lock_spec(parents, configs, seeds, lock_dir):
    """make_lock spec: one entry per (model, seed). Boards: the development boards of all seeds of that parent config."""
    models = []
    for config in configs:
        boards = [str(parents[(config, s)][1]) for s in seeds]
        for label, kind in KINDS[config]:
            for s in seeds:
                models.append({'name': f'{label}_seed{s}', 'kind': kind, 'boards': boards,
                               'checkpoint': str(lock_dir / f'{config}_seed{s}' / 'stage2_final.pt')})
    return {'models': models}


def find_official(input_dir):
    """Paths of the two official test CSVs, found by file name only (nothing is read). Checked before training."""
    found = {}
    for name in ('test_seen', 'test_unseen'):
        want = f'dpd_small_{name}.csv'
        hits = [p for p in Path(input_dir).rglob('*.csv') if p.name.lower() == want]
        assert len(hits) == 1, (f'expected one DPD_Small_{name}.csv in the inputs, found {hits}. '
                                'Add DPD_Small_test_seen.csv and DPD_Small_test_unseen.csv to the DPD dataset on Kaggle.')
        found[name] = hits[0]
    return found


def copy_official(found, data_dir):
    """Final mode only, after the lock is written: put the official test CSVs into the data root."""
    for name, path in found.items():
        shutil.copy2(path, Path(data_dir) / f'DPD_Small_{name}.csv')


def summarise(metrics_csv):
    """Mean and SD over seeds for each model group, catalogue, split and metric."""
    t = pd.read_csv(metrics_csv)
    t['group'] = t.model.str.replace(r'_seed\d+$', '', regex=True)
    g = t.groupby(['group', 'catalogue', 'split', 'metric']).value
    return pd.DataFrame({'mean': g.mean(), 'sd': g.std(ddof=1), 'seeds': g.count()}).reset_index()

## 4. Training images (the official test CSVs are added later, after the lock)

In [ ]:
train_csv = next(p for p in INPUT.rglob('*') if p.name.lower() == 'dpd_small_train.csv')
train = pd.read_csv(train_csv, header=None, names=COLUMNS)
assert len(train) == 8000, f'expected 8,000 rows, got {len(train)}'
digest = hashlib.sha256(train_csv.read_bytes()).hexdigest()
assert digest.startswith('9f1b729a'), f'train CSV hash {digest[:8]} does not match the split manifest'
first = train.filename.iloc[0]
image_src = next((d for d in [train_csv.parent, *train_csv.parent.parent.rglob('*')] if d.is_dir() and (d / first).is_file()), None)
assert image_src is not None, f'could not find {first} near {train_csv.parent}'
link = link_root(DATA_DIR, train_csv, image_src)
missing = [n for n in train.filename if not (link / n).is_file()]
assert not missing, f'{len(missing)} images missing, e.g. {missing[:3]}'
os.environ['DPD_ROOT'] = str(DATA_DIR)
print(f'8,000 training images linked from {image_src}')

if SMOKE:
    if OUT.exists():
        shutil.rmtree(OUT)  # smoke output only: a rerun starts clean
    link_root(STANDIN_DIR, train_csv, image_src)
    write_standins(STANDIN_DIR, load_config()['paths']['splits_dir'])
    print('smoke: stand-in test CSVs written from val_seen / val_unseen in', STANDIN_DIR)
else:
    OFFICIAL = find_official(INPUT)  # fail now, before training, if the test CSVs are missing
    print('official test CSVs found (not read yet):', *OFFICIAL.values())

## 5. Image cache for the 8,000 training images (a few minutes)

In [ ]:
sh(sys.executable, 'scripts/build_cache.py', '--verify', '--workers', 8, f'paths.cache_dir={CACHE_DIR}')

## 6. Parent checkpoints (epoch 10 of the development runs) and their development boards

In [ ]:
PARENT = {(c, s): find_parent(c, s, INPUT, PARENT_EPOCH) for c in PARENTS for s in SEEDS}
for (c, s), (pt, board) in PARENT.items():
    print(f'{c} seed {s}: {pt}')

## 7. Train: 2 more epochs per model on all 8,000 images. Each GPU takes the next waiting run

In [ ]:
RUNS = [f'{c}_seed{s}' for c in PARENTS for s in SEEDS]
FAILED = run_queue([stage2_job(c, s, PARENT[(c, s)][0], LOCK_DIR, SMOKE) for c in PARENTS for s in SEEDS], OUT / 'logs')
assert not FAILED, f'training failed for {FAILED}: read the logs in {OUT / "logs"}'
GATES = gate_table(LOCK_DIR, RUNS)
GATES.to_csv(OUT / 'stage2_gates.csv', index=False)
print(GATES.to_string(index=False))
ALL_GATES_PASS = bool((GATES.gate == 'PASS').all())
print('ALL GATES PASS' if ALL_GATES_PASS else 'A GATE FAILED: the run stops here and the test is not run. Read the loss table above.')

## 8. Fix the models (lock), only if every loss check passed. Nothing has read a test image yet

In [ ]:
assert ALL_GATES_PASS, 'a loss check failed: no lock is written and the test is not run'
spec_path = OUT / 'lock_spec.yaml'
spec_path.write_text(yaml.safe_dump(lock_spec(PARENT, PARENTS, SEEDS, LOCK_DIR)))
# smoke scores the stand-ins (dev validation sets), so it locks the dev catalogue
overrides = [f'paths.catalogue_final={load_config()["paths"]["catalogue_dev"]}'] if SMOKE else []
sh(sys.executable, 'scripts/make_lock.py', spec_path, LOCK_DIR / 'lock.json', *overrides)
LOCK = LOCK_DIR / 'lock.json'

## 9. The official test (once). Smoke runs the same path on the stand-in files

In [ ]:
if not SMOKE:
    copy_official(OFFICIAL, DATA_DIR)
    print('official test CSVs copied into', DATA_DIR)
RESULTS = OUT / ('standin_results' if SMOKE else 'test_results')
env = dict(os.environ, DPD_ROOT=str(STANDIN_DIR if SMOKE else DATA_DIR))
sh(sys.executable, 'scripts/final_test.py', LOCK, RESULTS, '--confirm-final-test',
   f'paths.cache_dir={CACHE_DIR}', f'loader.num_workers={WORKERS}', env=env)

## 10. Summary over seeds

In [ ]:
SUMMARY = summarise(RESULTS / 'final_metrics.csv')
SUMMARY.to_csv(RESULTS / 'final_summary.csv', index=False)
view = SUMMARY[SUMMARY.split == 'combined'].pivot_table(index=['group', 'catalogue'], columns='metric', values='mean')
print(view[['A_S', 'A_U', 'H', 'gap']].round(2).to_string())
if SMOKE:
    print('SMOKE finished: these numbers are on the development validation sets, not the test. Now set MODE = "final".')
else:
    print('FINAL finished: download final/test_results/.')